In [ ]:
import pacmap
import umap.umap_                       as umap 
import numpy                            as np
import pandas                           as pd

import seaborn                          as sns
import matplotlib.pyplot                as plt
import matplotlib.gridspec              as gridspec
import plotly.express                   as px
import plotly.graph_objects             as go

from tqdm                               import tqdm
from plotly.subplots                    import make_subplots
from sklearn.preprocessing              import StandardScaler, MinMaxScaler
from sklearn.cluster                    import DBSCAN

from utils.Preprocessing                import PreProcessData
from utils.plotFigures                  import plotResults, plotResultsPlotly, plotLabels, lineAndScatterPlot
from clustering.plotClustering          import ClusteringResults

In [ ]:
preData = PreProcessData()
preData.dataLoading(typeFile='h5')

df              = preData.splitTimeSeries()
dfHealthy       = preData.getHealthyState()

scaler          = MinMaxScaler()
scaledData      = preData.normalizeData(data   = df, 
                                        scaler = scaler) 
scaledHealthy   = preData.normalizeData(data   = dfHealthy, 
                                        scaler = scaler)

# Plot time series colored by labels

In [ ]:
def assign_labels(data, num_groups=4):
    n       = len(data)
    labels  = np.zeros(n, dtype=int)
    idx     = np.array_split(np.arange(n), num_groups)
    for i, group_idx in enumerate(idx):
        labels[group_idx] = i
    
    return labels

scaledData['label'] = scaledData.groupby('trajectory')['corriente(A)'].transform(assign_labels).astype('str')

In [ ]:
plotLabels(dataframe    = scaledData,
           variable     = 'Eje_piñón_aceite(%)')

# Dimensionality Reduction

In [ ]:
scaledData['period']    = scaledData['tiempo'].dt.strftime('%d-%m-%Y %H')
scaledHealthy['period'] = scaledHealthy['tiempo'].dt.strftime('%d-%m-%Y %H')

## Idea 1

In [ ]:
# Embedding space using healthy data
dfEfficiencyVar = scaledHealthy[preData.manipulatedCr1]

clusteringModels    = {'pacmap': DBSCAN(min_samples = 100,
                                        eps         = 1.2  )} 

dimReductionModels  = {'pacmap': pacmap.PaCMAP(n_components = 2, 
                                               random_state = 40, 
                                               save_tree    = True)}

EmbeddingHealthy = ClusteringResults(data                = scaledHealthy, 
                                     dataNormalized      = dfEfficiencyVar, 
                                     clusteringModels    = clusteringModels, 
                                     dimReductionModels  = dimReductionModels)

# Train dimensionality reduction model
EmbeddingHealthy.train()


# Embedding space for each trajectory
dfEfficiencyVar     = scaledData[preData.manipulatedCr1]
dimReductionModels  = EmbeddingHealthy.dimReductionModels

Embedding           = ClusteringResults(data                = scaledData, 
                                        dataNormalized      = dfEfficiencyVar, 
                                        clusteringModels    = clusteringModels, 
                                        dimReductionModels  = dimReductionModels)

# Trained embedding space is used
Embedding.embeddingTrain = EmbeddingHealthy.embeddingTrain


# Calculate embedding for all data per each trajectory
for trajectory in tqdm(Embedding.trajectories):
    Embedding.getResults(trajectory)

## Idea 2

In [ ]:
# Embedding space using labels
dfEfficiencyVar = scaledData[preData.manipulatedCr1]

clusteringModels    = {'umap':   DBSCAN(min_samples = 100,
                                        eps         = 2.2)} 

dimReductionModels  = {'umap':   umap.UMAP(n_components = 2,
                                           n_neighbors  = 15, 
                                           random_state = 40)}

SupervisedEmbedding = ClusteringResults(data                = scaledData, 
                               dataNormalized      = dfEfficiencyVar, 
                               clusteringModels    = clusteringModels, 
                               dimReductionModels  = dimReductionModels)

# Train dimensionality reduction model
SupervisedEmbedding.train(supervised=True)

# Get embedding space for each trajectory
for trajectory in tqdm(SupervisedEmbedding.trajectories):
    SupervisedEmbedding.getResults(trajectory)

# Results

## Seaborn

In [ ]:
trajectories    = Embedding.trajectories
for trajectory in trajectories:
    # Dataframes
    data1       = Embedding.clustersResults[trajectory]
    data2       = SupervisedEmbedding.clustersResults[trajectory]
    plotResults(trajectory  = trajectory, 
                data1       = data1, 
                data2       = data2,
                variable    = 'corriente(A)',
                palette     = 'Spectral')
    

## Plotly

In [ ]:
trajectory  = '15'
data1       = Embedding.clustersResults[trajectory]
data2       = SupervisedEmbedding.clustersResults[trajectory]
plotResultsPlotly(trajectory    = trajectory, 
                  data1         = data1, 
                  data2         = data2,
                  variable      = 'corriente(A)',
                  palette       = 'Spectral')